# 02 — Data Preprocessing

**Objective:** Clean the CRMLS sold-property data so it is ready for machine learning.  

**Steps we will follow:**
1. Load data and filter to Single Family Residences
2. Select the features (columns) we want to use
3. Handle missing values
4. Convert categorical (text) features to numbers
5. Split into training and testing sets by month
6. Normalize (scale) numerical features
7. Save the cleaned data to CSV files

## 1 — Imports

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2 — Load & Filter Data

We load all 6 months of CSV files and add a `month` column to each row  
so we know which month it came from (needed for the train/test split later).

In [4]:
# Path to our data folder
DATA_DIR = os.path.join(os.getcwd(), 'Data')


# Find all CRMLSSold CSV files, sorted by name (Jan first, Jun last)
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, 'CRMLSSold*.csv')))

# Load each file and add a 'month' column
dfs = []
for f in csv_files:
    temp = pd.read_csv(f, low_memory=False)
    
    # Extract "202401" from filename "CRMLSSold202401_filled.csv"
    filename = os.path.basename(f)
    month = filename.replace('CRMLSSold', '').replace('_filled.csv', '')
    temp['month'] = month
    
    dfs.append(temp)

df_raw = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(csv_files)} files — {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')

Loaded 6 files — 136,614 rows, 83 columns


In [5]:
# Filter to Residential / Single Family Residence only (as required by the task)
df = df_raw[
    (df_raw['PropertyType'] == 'Residential') &
    (df_raw['PropertySubType'] == 'SingleFamilyResidence')
].copy()

print(f'After filtering: {df.shape[0]:,} rows  ({df.shape[0] / df_raw.shape[0] * 100:.1f}% of raw data)')

After filtering: 68,641 rows  (50.2% of raw data)


## 3 — Select Features

Out of the 82 columns, we pick 10 features that are most likely to predict `ClosePrice`.  
We keep the selection small and simple for now — more features can be added in Week 6.

In [6]:
# Features we will use to predict ClosePrice
features = [
    'LivingArea',            # Square footage of living area
    'BedroomsTotal',         # Number of bedrooms
    'BathroomsTotalInteger', # Number of bathrooms
    'LotSizeSquareFeet',     # Lot size in square feet
    'YearBuilt',             # Year the property was built
    'GarageSpaces',          # Number of garage spaces
    'Stories',               # Number of stories
    'DaysOnMarket',          # Days on market before sale
    'City',                  # City name (text — will be encoded later)
    'PostalCode',            # ZIP code  (text — will be encoded later)
]

# The value we want to predict
target = 'ClosePrice'

# Keep only the columns we need + target + month (for splitting)
keep_cols = features + [target, 'month']
df = df[keep_cols].copy()

print(f'Selected {len(features)} features + target + month column')
print(f'Shape: {df.shape}')
df.head()

Selected 10 features + target + month column
Shape: (68641, 12)


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,Stories,DaysOnMarket,City,PostalCode,ClosePrice,month
5,1974.0,4.0,4.0,NaN,2023.0,2.0,2.0,33,National City,91950,815000.0,202401
9,1974.0,4.0,4.0,NaN,2023.0,2.0,2.0,228,National City,91950,810000.0,202401
28,3736.0,4.0,4.0,11219.0,2019.0,3.0,2.0,0,San Luis Obispo,93401,2100000.0,202401
29,2100.0,3.0,0.0,74487.6,1986.0,2.0,2.0,-36,Fort Bragg,95437,1950000.0,202401
31,2442.0,4.0,3.0,8712.0,1985.0,3.0,NaN,0,San Jose,95148,2340000.0,202401


## 4 — Handle Missing Values

**Strategy:**
- First, drop any rows where `ClosePrice` (our target) is missing — we can't train on those.
- For **numeric** columns: fill missing values with the **median** (the middle value).  
  Median is better than mean because it isn't affected by extreme outliers.
- For **categorical** (text) columns: fill missing values with `"Unknown"`.

In [7]:
# Let's see how many missing values each column has
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df):,}')

Missing values per column:
LivingArea                  36
BedroomsTotal                0
BathroomsTotalInteger       19
LotSizeSquareFeet         1159
YearBuilt                   49
GarageSpaces              2339
Stories                  10284
DaysOnMarket                 0
City                        90
PostalCode                   0
ClosePrice                   1
month                        0
dtype: int64

Total rows: 68,641


In [8]:
# Step 1: Drop rows where ClosePrice is missing (we need it to train the model)
before = len(df)
df = df.dropna(subset=[target])
print(f'Dropped {before - len(df)} rows with missing ClosePrice')

# Step 2: Define numeric vs categorical columns
numeric_features = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
    'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces',
    'Stories', 'DaysOnMarket'
]
categorical_features = ['City', 'PostalCode']

# Step 3: Fill missing numeric values with the median
for col in numeric_features:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'  Filled {col} missing values with median = {median_val}')

# Step 4: Fill missing categorical values with "Unknown"
for col in categorical_features:
    df[col] = df[col].fillna('Unknown')
    print(f'  Filled {col} missing values with "Unknown"')

# Verify: no more missing values
print(f'\nRemaining missing values: {df.isnull().sum().sum()}')

Dropped 1 rows with missing ClosePrice
  Filled LivingArea missing values with median = 1797.5
  Filled BedroomsTotal missing values with median = 3.0
  Filled BathroomsTotalInteger missing values with median = 2.0
  Filled LotSizeSquareFeet missing values with median = 7217.0
  Filled YearBuilt missing values with median = 1976.0
  Filled GarageSpaces missing values with median = 2.0
  Filled Stories missing values with median = 1.0
  Filled DaysOnMarket missing values with median = 13.0
  Filled City missing values with "Unknown"
  Filled PostalCode missing values with "Unknown"

Remaining missing values: 0


## 5 — Encode Categorical Features

Machine learning models need **numbers**, not text.  
We use `LabelEncoder` to assign a unique number to each city and postal code.

**Note:** Label encoding assigns arbitrary numbers (e.g., "Los Angeles" → 5, "San Diego" → 8).  
This works well for tree-based models (Decision Trees, Random Forests, etc.)  
but linear models might interpret these numbers as having an order.  
We'll keep it simple for now.

In [9]:
# Make sure PostalCode is a string before encoding
# (it might have been loaded as a number like 94401.0)
df['PostalCode'] = df['PostalCode'].astype(str)
df['PostalCode'] = df['PostalCode'].str.replace(r'\.0$', '', regex=True)

# Create label encoders
city_encoder = LabelEncoder()
postal_encoder = LabelEncoder()

# Fit and transform — converts text to numbers
df['City'] = city_encoder.fit_transform(df['City'])
df['PostalCode'] = postal_encoder.fit_transform(df['PostalCode'])

print(f'City: {len(city_encoder.classes_)} unique values encoded')
print(f'PostalCode: {len(postal_encoder.classes_)} unique values encoded')
print('\nFirst 5 rows after encoding:')
df.head()

City: 885 unique values encoded
PostalCode: 1669 unique values encoded

First 5 rows after encoding:


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,Stories,DaysOnMarket,City,PostalCode,ClosePrice,month
5,1974.0,4.0,4.0,7217.0,2023.0,2.0,2.0,33,502,316,815000.0,202401
9,1974.0,4.0,4.0,7217.0,2023.0,2.0,2.0,228,502,316,810000.0,202401
28,3736.0,4.0,4.0,11219.0,2019.0,3.0,2.0,0,697,698,2100000.0,202401
29,2100.0,3.0,0.0,74487.6,1986.0,2.0,2.0,-36,254,1488,1950000.0,202401
31,2442.0,4.0,3.0,8712.0,1985.0,3.0,1.0,0,692,1374,2340000.0,202401


## 6 — Train/Test Split by Month

Per the task requirements:
- **Test set** = the most recent month of data (June 2024)
- **Training set** = the X months immediately before June
- X is a **tunable parameter** — change `TRAIN_MONTHS` below to experiment

In [10]:
# ============================================================
# TUNABLE PARAMETER
# Change this number to use more or fewer training months.
# The maximum is 5 (Jan through May 2024).
# ============================================================
TRAIN_MONTHS = 5

# All available months in our data (sorted)
all_months = sorted(df['month'].unique())
print(f'Available months: {list(all_months)}')

# Test set = the last (most recent) month
test_month = all_months[-1]

# Training set = the TRAIN_MONTHS months right before the test month
train_months = all_months[-(TRAIN_MONTHS + 1):-1]

print(f'\nTraining months: {list(train_months)}')
print(f'Test month:      {test_month}')

# Split the data
train_df = df[df['month'].isin(train_months)].copy()
test_df  = df[df['month'] == test_month].copy()

# Drop the 'month' column — it was only needed for splitting, not for prediction
train_df = train_df.drop(columns=['month'])
test_df  = test_df.drop(columns=['month'])

total = len(train_df) + len(test_df)
print(f'\nTraining set: {len(train_df):,} rows ({len(train_df)/total*100:.1f}%)')
print(f'Test set:     {len(test_df):,} rows ({len(test_df)/total*100:.1f}%)')

Available months: ['202401', '202402', '202403', '202404', '202405', '202406']

Training months: ['202401', '202402', '202403', '202404', '202405']
Test month:      202406

Training set: 56,096 rows (81.7%)
Test set:     12,544 rows (18.3%)


## 7 — Normalize Numerical Features

`StandardScaler` transforms each numeric feature so it has **mean ≈ 0** and **std ≈ 1**.  
This helps some models (like Linear Regression) work better.

**Important:** We fit the scaler on the **training data only**, then apply (transform) it to  
both training and testing data. This prevents **data leakage** — the model should not  
learn anything from the test data during training.

In [11]:
scaler = StandardScaler()

# Fit on training data AND transform it
train_df[numeric_features] = scaler.fit_transform(train_df[numeric_features])

# Transform test data using the SAME scaler (do NOT fit again!)
test_df[numeric_features] = scaler.transform(test_df[numeric_features])

print('Training set stats after scaling (should be mean ≈ 0, std ≈ 1):')
print(train_df[numeric_features].describe().round(2))

Training set stats after scaling (should be mean ≈ 0, std ≈ 1):
       LivingArea  BedroomsTotal  BathroomsTotalInteger  LotSizeSquareFeet  \
count    56096.00       56096.00               56096.00           56096.00   
mean         0.00          -0.00                   0.00               0.00   
std          1.00           1.00                   1.00               1.00   
min         -1.97          -3.67                  -1.95              -0.02   
25%         -0.64          -0.50                  -0.45              -0.01   
50%         -0.22          -0.50                  -0.45              -0.01   
75%          0.38           0.56                   0.29              -0.01   
max         20.32          11.15                 129.27             193.50   

       YearBuilt  GarageSpaces   Stories  DaysOnMarket  
count   56096.00      56096.00  56096.00      56096.00  
mean        0.00         -0.00      0.00          0.00  
std         1.00          1.00      1.00          1.00  
min  

## 8 — Save Cleaned Data

We save three files:
- `train.csv` — training data (scaled)
- `test.csv` — test data (scaled)
- `cleaned_data.csv` — all data, encoded but NOT scaled (useful for exploration later)

In [12]:
# Save the scaled train and test sets
train_df.to_csv(os.path.join(DATA_DIR, 'train.csv'), index=False)
test_df.to_csv(os.path.join(DATA_DIR, 'test.csv'), index=False)

# Save the full cleaned dataset (encoded but NOT scaled)
# 'df' still has the original numeric values with encoded categories
df_clean = df.drop(columns=['month'])
df_clean.to_csv(os.path.join(DATA_DIR, 'cleaned_data.csv'), index=False)

print('Saved files to Data/ folder:')
print(f'  train.csv        — {len(train_df):,} rows (scaled)')
print(f'  test.csv         — {len(test_df):,} rows (scaled)')
print(f'  cleaned_data.csv — {len(df_clean):,} rows (encoded, not scaled)')

Saved files to Data/ folder:
  train.csv        — 56,096 rows (scaled)
  test.csv         — 12,544 rows (scaled)
  cleaned_data.csv — 68,640 rows (encoded, not scaled)


## 9 — Summary

### What we did:
1. **Loaded** 6 months of CRMLS data and filtered to Single Family Residences
2. **Selected** 10 features: LivingArea, Bedrooms, Bathrooms, LotSize, YearBuilt, GarageSpaces, Stories, DaysOnMarket, City, PostalCode
3. **Handled missing values**: median for numeric columns, "Unknown" for categorical columns
4. **Encoded** City and PostalCode from text to numbers using LabelEncoder
5. **Split** data by month: June 2024 = test, preceding months = training
6. **Normalized** numeric features using StandardScaler (fit on training data only)
7. **Saved** cleaned data to CSV files

### What's next (Week 4):
Train a **Linear Regression** baseline model using this cleaned data.